In [ ]:
import pandas as pd

df = pd.read_csv("nasa_feature.csv")

In [3]:
# 훈련용 80%, 테스트용 20%
from sklearn.model_selection import train_test_split

X = df[["tau_real"]]   # 입력
y = df["SOH"]          # 정답

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

In [4]:
# 선형회귀 모델 학습
from sklearn.linear_model import LinearRegression

model = LinearRegression()
model.fit(X_train, y_train)

LinearRegression()

In [5]:
# 예측
y_pred = model.predict(X_test)

In [6]:
from sklearn.metrics import mean_squared_error, r2_score

mse = mean_squared_error(y_test, y_pred)
r2 = r2_score(y_test, y_pred)

print("MSE:", mse)
print("R2:", r2)

MSE: 0.003313975678437561
R2: 0.6900732379010689


In [7]:
# tau_proxy 성능 확인
X = df[["tau_p4"]]
y = df["SOH"]
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_squared_error, r2_score
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

model = LinearRegression()
model.fit(X_train, y_train)

y_pred = model.predict(X_test)

print("MSE:", mean_squared_error(y_test, y_pred))
print("R2:", r2_score(y_test, y_pred))

MSE: 0.0005665308628161483
R2: 0.9470173915022501


In [8]:
# K-fold 검증, 5등분 다른 조합으로 검증
from sklearn.model_selection import cross_val_score

scores = cross_val_score(model, X, y, cv=5, scoring="r2")
print(scores)
print("Mean R2:", scores.mean())

[-8.41050707  0.50837664 -0.47519411 -0.38944397 -5.29362834]
Mean R2: -2.8120793688994103


In [9]:
# 시계열 데이터여서 시간 순으로 학습하도록 수정한 코드
from sklearn.model_selection import TimeSeriesSplit

tscv = TimeSeriesSplit(n_splits=5)

scores = cross_val_score(model, X, y, cv=tscv, scoring="r2")

print(scores)
print("Mean R2:", scores.mean())

[-0.91370683 -1.23864345 -0.64567092 -6.90302258 -7.96838328]
Mean R2: -3.5338854127677974


# Modeling 분석
- τ_real 기반 모델:
  - R² ≈ 0.69
  - 시상수와 SOH 간 기본적인 선형 관계 확인

- τ_proxy (tau_p4) 기반 모델:
  - R² ≈ 0.94
  - 실제 측정 불가능한 변수 없이도 높은 성능 확보

##  검증
- K-fold 및 TimeSeriesSplit 결과:
  - 성능이 크게 저하됨 (R² < 0)

- 원인:
  - 단일 배터리 셀 데이터 사용
  - Cycle 증가에 따른 단조 감소 구조
  - 데이터 다양성 부족 → 일반화 어려움

## 결론

- τ_proxy 구조는 매우 유효한 feature
- 실제 적용 가능성 있음
- 추가 데이터 확보 시 일반화 가능성 높음